# HippoRAG 1 ColBERTv2 Backend for `aichip_us`

This notebook builds only the ColBERTv2 backend. Prepare corpus/OpenIE artifacts locally first, upload them to Google Drive, then run these cells on a Colab GPU runtime. It is designed for Colab free plan with checkpointed outputs.

In [ ]:
import os, subprocess, sys

def run(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, cwd=cwd, shell=True, check=check)

run('nvidia-smi')
DATASET = 'aichip_us'
DRIVE_DIR = '/content/drive/MyDrive/hipporag_colbert_aichip_us'
WORK_DIR = '/content/HippoRAG'
ARCHIVE_NAME = f'{DATASET}_colbertv2_artifacts.tar.gz'
os.environ.setdefault('OPENIE_MODEL', 'gpt-5.5')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive dir:', DRIVE_DIR)


In [ ]:
if not os.path.exists(WORK_DIR):
    run(f'git clone --branch legacy --single-branch https://github.com/OSU-NLP-Group/HippoRAG.git {WORK_DIR}')
else:
    print('Repo already exists:', WORK_DIR)
run('git rev-parse --short HEAD', cwd=WORK_DIR)


In [ ]:
# Install ColBERT/HippoRAG legacy dependencies. Re-running this cell is safe.
run('pip install -U pip')
run('pip install colbert-ai==0.2.19 langchain langchain-openai langchain-community langchain-together openai python-dotenv tiktoken tqdm ipdb thefuzz rank-bm25 pytrec_eval python-igraph pandas scipy transformers==4.37.2 tokenizers==0.15.2')
run('python - <<\'PY\'\nimport transformers\nprint("transformers", transformers.__version__, "has AdamW", hasattr(transformers, "AdamW"))\nPY')

# Colab images change. Try GPU FAISS first, then fall back to CPU FAISS if needed.
try:
    run('pip install faiss-gpu-cu12', check=True)
except Exception:
    run('pip install faiss-cpu', check=True)


In [ ]:
ckpt_dir = os.path.join(WORK_DIR, 'exp', 'colbertv2.0')
if not os.path.exists(ckpt_dir):
    run('mkdir -p exp', cwd=WORK_DIR)
    run('wget -nc https://downloads.cs.stanford.edu/nlp/data/colbert/colbertv2/colbertv2.0.tar.gz -O exp/colbertv2.0.tar.gz', cwd=WORK_DIR)
    run('tar -xzf exp/colbertv2.0.tar.gz -C exp', cwd=WORK_DIR)
else:
    print('Checkpoint already exists:', ckpt_dir)


In [ ]:
# Copy local artifacts from Drive to the Colab local disk.
# Accepts either files directly under DRIVE_DIR, a nested colab_inputs_* folder, or colab_inputs_*.tar.gz.
import glob, shutil
run('mkdir -p data output data/lm_vectors/colbert /content/hipporag_colab_inputs')
archive = os.path.join(DRIVE_DIR, f'colab_inputs_{DATASET}.tar.gz')
nested_dir = os.path.join(DRIVE_DIR, f'colab_inputs_{DATASET}')
if os.path.exists(archive):
    run(f'tar -xzf {archive} -C /content/hipporag_colab_inputs')
    INPUT_DIR = os.path.join('/content/hipporag_colab_inputs', f'colab_inputs_{DATASET}')
elif os.path.exists(nested_dir):
    INPUT_DIR = nested_dir
else:
    INPUT_DIR = DRIVE_DIR
print('Using input dir:', INPUT_DIR)
required = [f'{DATASET}_corpus.json', f'openie_{DATASET}_results_ner_gpt-5.5_200.json']
missing = [name for name in required if not os.path.exists(os.path.join(INPUT_DIR, name))]
if missing:
    print('Available files in DRIVE_DIR:')
    run(f'find {DRIVE_DIR} -maxdepth 2 -type f | sort | sed -n "1,120p"', check=False)
    raise FileNotFoundError(f'Missing required Colab input files: {missing}')
shutil.copy2(os.path.join(INPUT_DIR, f'{DATASET}_corpus.json'), os.path.join(WORK_DIR, 'data', f'{DATASET}_corpus.json'))
for name in [f'{DATASET}.json', f'{DATASET}_id_map.json', f'{DATASET}_value_scores.json']:
    src = os.path.join(INPUT_DIR, name)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(WORK_DIR, 'data', name))
for src in glob.glob(os.path.join(INPUT_DIR, f'openie_{DATASET}_results_*.json')):
    shutil.copy2(src, os.path.join(WORK_DIR, 'output', os.path.basename(src)))
ner_src = os.path.join(INPUT_DIR, f'{DATASET}_queries.named_entity_output.tsv')
if os.path.exists(ner_src):
    shutil.copy2(ner_src, os.path.join(WORK_DIR, 'output', os.path.basename(ner_src)))
run('ls -lh data output | sed -n "1,160p"', cwd=WORK_DIR)


In [ ]:
# Query NER should be uploaded from the local codex-proxy OpenIE run.
# Colab should only build the ColBERTv2 graph/index from precomputed artifacts.
if not os.path.exists(os.path.join(WORK_DIR, 'output', f'{DATASET}_queries.named_entity_output.tsv')):
    print('Query NER file missing. Generate it locally with codex-proxy, then re-upload the Colab input tarball.')
    # run(f'python src/named_entity_extraction_parallel.py --dataset {DATASET} --llm openai --model_name "$OPENIE_MODEL"', cwd=WORK_DIR)
else:
    print('Query NER exists.')


In [ ]:
# First create_graph pass: generate query_to_kb.tsv and kb_to_kb.tsv.
run(f'python src/create_graph.py --dataset {DATASET} --model_name colbertv2 --extraction_model "$OPENIE_MODEL" --threshold 0.8 --extraction_type ner --cosine_sim_edges', cwd=WORK_DIR)
run('ls -lh output/*_to_kb.tsv output/rel_kb_to_kb.tsv', cwd=WORK_DIR)


In [ ]:
# ColBERT KNN. Checkpoint outputs are copied to Drive after each major step.
# Patch legacy script to skip empty phrase rows that pandas reads as NaN.
knn_path = os.path.join(WORK_DIR, 'src', 'colbertv2_knn.py')
knn_src = open(knn_path).read()
old = "string_df = pd.read_csv(string_filename, sep='\\t')\n    string_df.strings = [processing_phrases(str(s)) for s in string_df.strings]"
new = "string_df = pd.read_csv(string_filename, sep='\\t')\n    string_df = string_df.dropna(subset=['strings'])\n    string_df.strings = [processing_phrases(str(s)) for s in string_df.strings]\n    string_df = string_df[string_df.strings.str.strip() != '']"
if old in knn_src:
    open(knn_path, 'w').write(knn_src.replace(old, new))
else:
    print('KNN script already patched or source changed.')
run('python - <<\'PY\'\nimport pandas as pd\nfor name in ["kb_to_kb", "query_to_kb"]:\n    df = pd.read_csv(f"output/{name}.tsv", sep="\\t")\n    print(name, "rows", len(df), "nan", int(df["strings"].isna().sum()))\nPY', cwd=WORK_DIR)
run('bash -o pipefail -c "python -u src/colbertv2_knn.py --filename output/kb_to_kb.tsv 2>&1 | tee /content/colbert_knn_kb_to_kb.log"', cwd=WORK_DIR)
run('bash -o pipefail -c "python -u src/colbertv2_knn.py --filename output/query_to_kb.tsv 2>&1 | tee /content/colbert_knn_query_to_kb.log"', cwd=WORK_DIR)
run(f'mkdir -p {DRIVE_DIR}/checkpoints')
run(f'cp /content/colbert_knn_*.log {DRIVE_DIR}/checkpoints/ || true')
run(f'tar -czf {DRIVE_DIR}/checkpoints/{DATASET}_colbert_knn.tar.gz data/lm_vectors/colbert output', cwd=WORK_DIR)


In [ ]:
# Second create_graph pass: build the ColBERT synonym-edge graph artifacts.
run(f'python src/create_graph.py --dataset {DATASET} --model_name colbertv2 --extraction_model "$OPENIE_MODEL" --threshold 0.8 --create_graph --extraction_type ner --cosine_sim_edges', cwd=WORK_DIR)
run('ls -lh output | grep -i "aichip_us.*graph" | head -50', cwd=WORK_DIR)


In [ ]:
# Build phrase and corpus ColBERT indices.
# Patch the legacy ColBERT/HippoRAG files first; this cell is safe to rerun.
import os, shlex
patch_script = os.path.join(DRIVE_DIR, 'colbertv2_colab_runtime_patch.py')
if not os.path.exists(patch_script):
    raise FileNotFoundError(f'Missing Colab runtime patch: {patch_script}')
run(f'cp {shlex.quote(patch_script)} {shlex.quote(os.path.join(WORK_DIR, "colbertv2_colab_runtime_patch.py"))}')
run('python colbertv2_colab_runtime_patch.py', cwd=WORK_DIR)

extraction_suffix = 'ner' if os.environ.get('OPENIE_MODEL') == 'gpt-3.5-turbo-1106' else 'ner_' + os.environ.get('OPENIE_MODEL', 'gpt-5.5').replace('/', '_')
phrase_path = f'output/{DATASET}_facts_and_sim_graph_phrase_dict_ents_only_lower_preprocess_{extraction_suffix}.v3.subset.p'
if not os.path.exists(os.path.join(WORK_DIR, phrase_path)):
    run(f'find output -maxdepth 1 -name "{DATASET}_*graph_phrase_dict*" -print', cwd=WORK_DIR, check=False)
    raise FileNotFoundError(phrase_path)
run(f'bash -o pipefail -c "python -u src/colbertv2_indexing.py --dataset {DATASET} --phrase {phrase_path} --corpus data/{DATASET}_corpus.json 2>&1 | tee /content/colbert_indexing.log"', cwd=WORK_DIR)
run(f'mkdir -p {DRIVE_DIR}/checkpoints')
run(f'cp /content/colbert_indexing.log {DRIVE_DIR}/checkpoints/ || true')
run(f'tar -czf {DRIVE_DIR}/checkpoints/{DATASET}_colbert_indices.tar.gz data/lm_vectors/colbert', cwd=WORK_DIR)


In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/hipporag_colbert_aichip_us"
WORK_DIR = "/content/HippoRAG"

!cp -f "$DRIVE_DIR/colbertv2_colab_runtime_patch.py" "$WORK_DIR/colbertv2_colab_runtime_patch.py"
!grep -n "overwrite='reuse'" "$WORK_DIR/colbertv2_colab_runtime_patch.py" | head
!cd "$WORK_DIR" && python colbertv2_colab_runtime_patch.py
!grep -n "overwrite='reuse'" "$WORK_DIR/src/hipporag.py"
!cd "$WORK_DIR" && python -m py_compile src/named_entity_extraction_parallel.py src/openie_with_retrieval_option_parallel.py src/colbertv2_indexing.py src/hipporag.py

In [ ]:
%run -i /content/drive/MyDrive/hipporag_colbert_aichip_us/smoke_cell_patch.py

In [ ]:
!tail -200 /content/smoke_colbert.log

In [ ]:
# Final archive for download/sync back to the local repo.
run(f'tar -czf {DRIVE_DIR}/{ARCHIVE_NAME} output data/lm_vectors/colbert', cwd=WORK_DIR)
print('Wrote:', os.path.join(DRIVE_DIR, ARCHIVE_NAME))
